# US Aviation — Build Dashboard Prediction Dataset
### Random Forest Model

Loads the trained Random Forest model and generates a prediction dataset
on the held-out test set (20% of `flights_model_ready_3class.csv`).

Output columns added on top of the original features:
| Column | Description |
|---|---|
| `actual_class` | True delay class (0/1/2) |
| `predicted_class` | RF prediction |
| `actual_label` | Human-readable actual label |
| `predicted_label` | Human-readable predicted label |
| `prob_class_0/1/2` | Per-class predicted probabilities |
| `is_correct` | 1 if prediction matches actual |
| `prediction_confidence` | Max probability across classes |

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 1 — GPU Check & Install cuML
cuML is required to load the Random Forest model saved by the training notebook.

In [2]:
import subprocess, sys

def _has_gpu():
    try:
        subprocess.run(["nvidia-smi"], check=True, capture_output=True)
        return True
    except Exception:
        return False

USE_GPU = _has_gpu()

if USE_GPU:
    print("GPU detected — installing cuML…")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet",
         "--extra-index-url", "https://pypi.nvidia.com",
         "cuml-cu12", "cudf-cu12"],
        check=True
    )
    print("cuML installed.")
else:
    print("No GPU — will load RF as sklearn model (CPU).")

GPU detected — installing cuML…
cuML installed.


In [3]:
import json, joblib, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

warnings.filterwarnings("ignore")
print("Libraries loaded.")

Libraries loaded.


## Step 2 — Configure Paths

In [4]:

BASE          = Path("/content/drive/MyDrive/aviation")
MODEL_READY   = BASE / "flights_model_ready_3class.csv"
MODELS_DIR    = BASE / "models"
DASHBOARD_CSV = BASE / "dashboard_predictions_3class.csv"
REPORT_JSON   = BASE / "dashboard_data_report.json"


TARGET_COL   = "delay_class_3"
RANDOM_STATE = 42
CLASS_LABELS = {0: "<=15 min", 1: "16-60 min", 2: ">60 min"}

print("Checking required files:")
for p in [MODEL_READY,
          MODELS_DIR / "random_forest_3class.joblib",
          MODELS_DIR / "ordinal_encoder.joblib",
          MODELS_DIR / "standard_scaler.joblib"]:
    print(f"  [{'OK' if p.exists() else 'MISSING'}]  {p.name}")

Checking required files:
  [OK]  flights_model_ready_3class.csv
  [OK]  random_forest_3class.joblib
  [OK]  ordinal_encoder.joblib
  [OK]  standard_scaler.joblib


## Step 3 — Load Data & Preprocessors

In [5]:
print("Loading model-ready dataset…")
df = pd.read_csv(MODEL_READY, low_memory=False)
print(f"Shape: {df.shape}")

print("\nLoading preprocessors…")
ord_enc = joblib.load(MODELS_DIR / "ordinal_encoder.joblib")
scaler  = joblib.load(MODELS_DIR / "standard_scaler.joblib")
print("OrdinalEncoder and StandardScaler loaded.")

Loading model-ready dataset…
Shape: (2912112, 23)

Loading preprocessors…
OrdinalEncoder and StandardScaler loaded.


## Step 4 — Preprocess & Split

In [6]:
DROP_COLS = ["FL_DATE", "AIRLINE_CODE", "route"]
df_feat   = df.drop(columns=[c for c in DROP_COLS if c in df.columns], errors="ignore")

X_raw = df_feat.drop(columns=[TARGET_COL])
y     = df_feat[TARGET_COL].astype(int).values

cat_cols      = X_raw.select_dtypes(include=["object", "string"]).columns.tolist()
num_cols      = X_raw.select_dtypes(include=["number"]).columns.tolist()

X_cat = ord_enc.transform(X_raw[cat_cols]).astype("float32")
X_num = scaler.transform(X_raw[num_cols]).astype("float32")
X     = np.hstack([X_num, X_cat])

# Same 80/20 stratified split as training
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
_, X_test_raw, _, y_test_raw = train_test_split(
    X_raw, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Test set: {X_test.shape[0]:,} rows  ({X_test.shape[1]} features)")

Test set: 582,423 rows  (19 features)


## Step 5 — Load Random Forest & Run Inference

In [7]:
print("Loading Random Forest model…")
rf_model = joblib.load(MODELS_DIR / "random_forest_3class.joblib")
print(f"Loaded: {type(rf_model).__name__}")

print("\nRunning predictions…")
if USE_GPU:
    import cupy as cp
    X_in   = cp.array(X_test)
    preds  = rf_model.predict(X_in)
    probs  = rf_model.predict_proba(X_in)
    # cuML returns cupy arrays — use .get() to bring back to numpy
    preds  = preds.get().astype(int) if hasattr(preds, "get") else np.array(preds).astype(int)
    probs  = probs.get()             if hasattr(probs, "get") else np.array(probs)
else:
    preds  = rf_model.predict(X_test).astype(int)
    probs  = rf_model.predict_proba(X_test)

acc = accuracy_score(y_test, preds)
print(f"Predictions complete.")
print(f"Test accuracy: {acc:.4f}")

Loading Random Forest model…
Loaded: RandomForestClassifier

Running predictions…
Predictions complete.
Test accuracy: 0.7750


## Step 6 — Assemble Output DataFrame

In [8]:
out = X_test_raw.copy().reset_index(drop=True)

out["actual_class"]          = y_test_raw
out["predicted_class"]       = preds
out["actual_label"]          = out["actual_class"].map(CLASS_LABELS)
out["predicted_label"]       = out["predicted_class"].map(CLASS_LABELS)
out["prob_class_0"]          = probs[:, 0].round(4)
out["prob_class_1"]          = probs[:, 1].round(4)
out["prob_class_2"]          = probs[:, 2].round(4)
out["is_correct"]            = (out["actual_class"] == out["predicted_class"]).astype(int)
out["prediction_confidence"] = probs.max(axis=1).round(4)

print(f"Output shape: {out.shape}")
out[["actual_label", "predicted_label", "is_correct", "prediction_confidence"]].head(6)

Output shape: (582423, 28)


,actual_label,predicted_label,is_correct,prediction_confidence
0,16-60 min,>60 min,0,0.4029
1,<=15 min,<=15 min,1,0.3907
2,<=15 min,<=15 min,1,0.6246
3,<=15 min,>60 min,0,0.4053
4,<=15 min,>60 min,0,0.4443
5,<=15 min,<=15 min,1,0.6849


## Step 7 — Summary Stats

In [9]:
print(f"Overall accuracy   : {out['is_correct'].mean():.4f}")
print(f"Avg confidence     : {out['prediction_confidence'].mean():.4f}")

print("\nPer-class accuracy:")
for cls, label in CLASS_LABELS.items():
    mask    = out["actual_class"] == cls
    cls_acc = out.loc[mask, "is_correct"].mean()
    print(f"  Class {cls} ({label:10s}): {cls_acc:.4f}  (n={mask.sum():,})")

print("\nClassification report:")
print(classification_report(y_test, preds,
      target_names=list(CLASS_LABELS.values()), zero_division=0))

Overall accuracy   : 0.7750
Avg confidence     : 0.5731

Per-class accuracy:
  Class 0 (<=15 min  ): 0.9012  (n=480,319)
  Class 1 (16-60 min ): 0.1614  (n=66,789)
  Class 2 (>60 min   ): 0.2189  (n=35,315)

Classification report:
              precision    recall  f1-score   support

    <=15 min       0.86      0.90      0.88    480319
   16-60 min       0.30      0.16      0.21     66789
     >60 min       0.18      0.22      0.20     35315

    accuracy                           0.77    582423
   macro avg       0.45      0.43      0.43    582423
weighted avg       0.75      0.77      0.76    582423



## Step 8 — Save to Drive

In [10]:
print("Saving dashboard CSV…")
out.to_csv(DASHBOARD_CSV, index=False)
print(f"Saved → {DASHBOARD_CSV}")
print(f"  {out.shape[0]:,} rows  ×  {out.shape[1]} columns")

report = {
    "input_data":       str(MODEL_READY),
    "output_csv":       str(DASHBOARD_CSV),
    "model_used":       "random_forest_3class.joblib",
    "row_count":        int(len(out)),
    "column_count":     int(len(out.columns)),
    "columns":          list(out.columns),
    "overall_accuracy": float(out["is_correct"].mean()),
    "avg_confidence":   float(out["prediction_confidence"].mean()),
    "class_accuracy": {
        str(cls): float(out.loc[out["actual_class"] == cls, "is_correct"].mean())
        for cls in CLASS_LABELS
    },
}

REPORT_JSON.write_text(json.dumps(report, indent=2), encoding="utf-8")
print(f"Saved → {REPORT_JSON}")

Saving dashboard CSV…
Saved → /content/drive/MyDrive/aviation/dashboard_predictions_3class.csv
  582,423 rows  ×  28 columns
Saved → /content/drive/MyDrive/aviation/dashboard_data_report.json


## Step 7 — Quick Stats Summary

In [11]:
print(f"Overall accuracy : {out['is_correct'].mean():.4f}")
print(f"Avg confidence   : {out['prediction_confidence'].mean():.4f}")

print("\nPrediction distribution:")
pred_dist = out["predicted_label"].value_counts()
for label, count in pred_dist.items():
    print(f"  {label:12s}: {count:>8,}  ({count/len(out)*100:.1f}%)")

print("\nActual distribution:")
act_dist = out["actual_label"].value_counts()
for label, count in act_dist.items():
    print(f"  {label:12s}: {count:>8,}  ({count/len(out)*100:.1f}%)")

print("\nAccuracy by class:")
for cls, label in CLASS_LABELS.items():
    mask = out["actual_class"] == cls
    cls_acc = out.loc[mask, "is_correct"].mean()
    n = mask.sum()
    print(f"  Class {cls} ({label:10s}): {cls_acc:.4f}  (n={n:,})")

Overall accuracy : 0.7750
Avg confidence   : 0.5731

Prediction distribution:
  <=15 min    :  503,150  (86.4%)
  >60 min     :   43,600  (7.5%)
  16-60 min   :   35,673  (6.1%)

Actual distribution:
  <=15 min    :  480,319  (82.5%)
  16-60 min   :   66,789  (11.5%)
  >60 min     :   35,315  (6.1%)

Accuracy by class:
  Class 0 (<=15 min  ): 0.9012  (n=480,319)
  Class 1 (16-60 min ): 0.1614  (n=66,789)
  Class 2 (>60 min   ): 0.2189  (n=35,315)


## Step 8 — Save to Drive

In [13]:
print("Saving dashboard CSV…")
out.to_csv(DASHBOARD_CSV, index=False)
print(f"Saved → {DASHBOARD_CSV}")
print(f"  {out.shape[0]:,} rows  ×  {out.shape[1]} columns")

report = {
    "input_data":       str(MODEL_READY),
    "output_csv":       str(DASHBOARD_CSV),
    "model_used":       "random_forest_3class.joblib",
    "row_count":        int(len(out)),
    "column_count":     int(len(out.columns)),
    "columns":          list(out.columns),
    "overall_accuracy": float(out["is_correct"].mean()),
    "avg_confidence":   float(out["prediction_confidence"].mean()),
    "class_accuracy": {
        str(cls): float(out.loc[out["actual_class"] == cls, "is_correct"].mean())
        for cls in CLASS_LABELS
    },
}

REPORT_JSON.write_text(json.dumps(report, indent=2), encoding="utf-8")
print(f"Saved → {REPORT_JSON}")

Saving dashboard CSV…
Saved → /content/drive/MyDrive/aviation/dashboard_predictions_3class.csv
  582,423 rows  ×  28 columns
Saved → /content/drive/MyDrive/aviation/dashboard_data_report.json


## Done

Files saved to Google Drive:

| File | Contents |
|---|---|
| `dashboard_predictions_3class.csv` | Test set with RF predictions & probabilities |
| `dashboard_data_report.json` | Accuracy, confidence, per-class breakdown |

Column groups in the output CSV:

**Original features** (19 cols): AIRLINE, ORIGIN, DEST, ORIGIN_STATE, DEST_STATE,
DISTANCE, CRS_DEP_TIME_MIN, CRS_ARR_TIME_MIN, CRS_ELAPSED_TIME, SCHED_BLOCK_MINS,
year, month, day, day_of_week, is_weekend, dep_hour, arr_hour, dep_time_bucket, season

**Prediction columns** (9 cols):
actual_class, predicted_class, actual_label, predicted_label,
prob_class_0, prob_class_1, prob_class_2, is_correct, prediction_confidence